In [0]:
import requests
from pyspark.sql.functions import current_timestamp

In [0]:
dbutils.widgets.text("API_KEY", "")
dbutils.widgets.text("SUBGRAPH_ID", "")
dbutils.widgets.text("pool_id", "")

In [0]:
API_KEY = dbutils.widgets.get("API_KEY")
SUBGRAPH_ID = dbutils.widgets.get("SUBGRAPH_ID")
query_variables= {
    "pool_id": dbutils.widgets.get("pool_id")
}

In [0]:
SUBGRAPH_UNIV3_URL = f"https://gateway.thegraph.com/api/{API_KEY}/subgraphs/id/{SUBGRAPH_ID}"

In [0]:
tick_query = """
query TicksQuery($pool_id: String!) {
  ticks(
    where: {
      pool: $pool_id
    }
    first: 1000
  ){
    id
    poolAddress
    tickIdx
    pool
    volumeToken0
    volumeToken1
    volumeUSD
    price0
    price1
    liquidityGross
    liquidityNet
    feesUSD
    collectedFeesToken0
    collectedFeesToken1
    collectedFeesUSD
    createdAtTimestamp
    createdAtBlockNumber
  }
}
"""

In [0]:
response = requests.post(SUBGRAPH_UNIV3_URL, json={"query": tick_query, "variables": query_variables})

In [0]:
df = spark.createDataFrame(response.json()["data"]["ticks"])
df = df.withColumn("_load_ts", current_timestamp())

In [0]:
display(df)

In [0]:
df.write.format("delta").mode("append").saveAsTable("uniswap.bronze.ticks")